# Reproduction: fork detection and base-trunk height

This notebook reproduces the evaluation reported in the manuscript: the
standalone fork-detection accuracy on manually corrected stems, the fork
detection for the two end-to-end pipelines, and the base-trunk height agreement
against the MLS reference.

All cells are checked in with their outputs, so the tables and figures are
visible without running anything. Rerunning requires the external tools and the
model weights described in the cell below, which covers the upstream setup.

## Upstream pipeline

<img src="figures/pipeline.png" width="600">

Upstream pipeline involve multiple steps and external tools:

The process starts with a raw ALS crop (e.g, ....las file in sample data)

#### 1. Denoise the raw ALS crop (statistical outlier removal)

```bash
python src/denoise_sor.py sample_data/<tree>.las sample_data/<tree>_filtered.las
```

`--nb-neighbors` (default 6) sets how many neighbours estimate each point's
local mean distance; `--std-ratio` (default 2.0) is the cut-off in standard
deviations, with lower values removing more points.


#### 2. Instance Segmentation with ForestFormer3D

Pipeline 1 — FOR-instanceV2 weights (public).
Follow the repository for run instructions

Pipeline 2 — FGI-EMIT weights (on request).
Contact the authors for code and model weights


#### 3. Select the centre instance for each plot (only one tree per ALS crop is evaluated)

Each crop covers several trees, so one instance must be singled out. This step
needs the manually cropped reference tree as a second input: its bounding box
defines the region of interest, and the largest predicted instance whose points fall
inside that box is kept.

```bash
python src/select_centre_instance.py \
    <segmented_crop>.laz <reference_tree>.las <selected_instance>.las
```


#### 4. Stem-point Extraction

Pipeline 1 obtains this from ForestFormer3D in step 2 itself. No additional step required.
step 2, so no separate step is needed. 

Pipeline 2 runs ResNet16FPN. Contact the authors for code and model weights.


#### 5. Prepare AdTree input (keep stem points, write `.xyz`):


The stem points are written to a min-normalised XYZ text file, the input format
AdTree reads:

```bash
python src/las_to_xyz.py <stem_points>.las <tree>.xyz
```

#### 6. Skeletonisation with AdTree (one `<name>_skeleton.ply` per tree):

AdTree is an external tool (build it from https://github.com/tudelft3d/AdTree).
Run its command-line binary on the `.xyz` from step 5, writing the skeleton to
an output folder:

```bash
./AdTree <tree>.xyz <output_folder>/ -s
```


#### 7. Fork detection

The skeletons from step 6 are the input to the fork-detection tool. Run it on
the folder of skeletons with the main CLI:

```bash
python ../scripts/fork_detect.py --folder <skeletons_folder>/ --out results.csv
```

This writes the fork decision and base-trunk height for each tree. See the
top-level `README.md` for the tool's options and output columns.

## Evaluation on the dataset used in the paper

In [1]:
import os
from pathlib import Path
import sys
import warnings

os.environ["EVO_DATA_ROOT"] = "/home/arun/PointClouds/SilvaFennica/code/test_data"

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import config
from runner import run_dataset
from reference import load_mls_reference
from metrics import (
    fork_detection_table, print_fork_detection,
    funnel_row, print_funnel_row,
    pair_estimates_with_reference, height_metrics, print_height_metrics,
)

### 1. Standalone fork detection

Fork/non-fork accuracy on the manually corrected stems. This isolates the
detector from upstream segmentation error and is the upper bound on detector
quality.

In [ ]:
warnings.filterwarnings("ignore", message=r".*duplicate XYZ merged.*")

standalone = fork_detection_table(
    run_dataset("manual_forked") + run_dataset("manual_unforked"))
print_fork_detection(standalone, title="Manually corrected")

def standalone_df(t):
    rows = [
        ("Forked",     t["n_forked"],   t["tp"],           t["acc_forked"]),
        ("Non-forked", t["n_unforked"], t["tn"],           t["acc_unforked"]),
        ("Overall",    t["n_total"],    t["tp"] + t["tn"], t["acc_overall"]),
    ]
    return pd.DataFrame(
        [(c, n, f"{k} ({_pct(a)})") for c, n, k, a in rows],
        columns=["Class", "Trees", "Classified correctly"],
    ).set_index("Class")

standalone_df(standalone)


=== Manually corrected ===
  forked   :  168 /  175   (96.0%)
  unforked :  158 /  159   (99.4%)
  overall  :  326 /  334   (97.6%)


### 2. End-to-end pipelines

For each end-to-end pipeline, this reports how many trees are correctly
segmented at the instance and semantic stages, and, among those, how many are
then given the correct fork/non-fork label. The instance-stage counts are
expressed as percentages of `config.TOTAL_FORKED` and `config.TOTAL_UNFORKED`;
the later stages are relative to the preceding stage.

In [ ]:
PIPELINES = {
    "ForestFormer3D (FOR-instanceV2)": {
        "inst": ("p1_INST_forked", "p1_INST_unforked"),
        "sem":  ("p1_SEM_forked",  "p1_SEM_unforked"),
    },
    "ForestFormer3D (FGI-EMIT) + ResNet16FPN": {
        "inst": ("p2_INST_forked", "p2_INST_unforked"),
        "sem":  ("p2_SEM_forked",  "p2_SEM_unforked"),
    },
}

def stage_table(forked_ds, unforked_ds):
    return fork_detection_table(run_dataset(forked_ds) + run_dataset(unforked_ds))


funnel_rows = []
for pipeline, stages in PIPELINES.items():
    inst_tab = stage_table(*stages["inst"])
    sem_tab  = stage_table(*stages["sem"])
    print(f"\n=== {pipeline} ===")
    for cls, total in (("forked", config.TOTAL_FORKED),
                       ("unforked", config.TOTAL_UNFORKED)):
        row = funnel_row(cls, inst_tab, sem_tab, total)
        row["pipeline"] = pipeline
        funnel_rows.append(row)
        print_funnel_row(row, pipeline=pipeline)

[p1_INST_forked] running 133 skeletons from /home/arun/PointClouds/SilvaFennica/code/test_data/ForInstanceV2/ValidInstances/Forked
  [10/133]  last=ALS_gridID_11_8_plotID_na_treeID_177_cropID_11_8_skeleton.ply  status=ok
  [20/133]  last=ALS_gridID_15_17_plotID_na_treeID_60_cropID_15_17_15_18_skeleton.ply  status=ok
  [30/133]  last=ALS_gridID_17_19_plotID_na_treeID_40_cropID_17_19_17_20_skeleton.ply  status=ok
  [40/133]  last=ALS_gridID_17_6_plotID_na_treeID_112_cropID_17_6_17_7_18_6_18_7_skeleton.ply  status=ok
  [50/133]  last=ALS_gridID_18_18_plotID_na_treeID_369_cropID_18_17_18_18_19_17_19_18_skeleton.ply  status=ok
  [60/133]  last=ALS_gridID_19_16_plotID_5_treeID_4_8_cropID_19_16_19_17_skeleton.ply  status=ok
  [70/133]  last=ALS_gridID_19_18_plotID_6_treeID_30_cropID_19_18_20_18_skeleton.ply  status=ok
  [80/133]  last=ALS_gridID_19_8_plotID_3_treeID_6_cropID_19_8_20_8_skeleton.ply  status=ok
  [90/133]  last=ALS_gridID_1_2_plotID_1_treeID_12_cropID_1_2_2_2_skeleton.ply  statu

In [10]:
import pandas as pd
from IPython.display import display

def _pct(x):
    return "-" if x != x else f"{100*x:.1f}%"   # x != x catches NaN

CLASS_LABEL = {"forked": "Forked", "unforked": "Non-forked"}

Section 1 — standalone accuracy (Table 1):

In [ ]:
standalone = fork_detection_table(
    run_dataset("manual_forked") + run_dataset("manual_unforked"))
print_fork_detection(standalone, title="Manually corrected")

def standalone_df(t):
    rows = [
        ("Forked",     t["n_forked"],   t["tp"],           t["acc_forked"]),
        ("Non-forked", t["n_unforked"], t["tn"],           t["acc_unforked"]),
        ("Overall",    t["n_total"],    t["tp"] + t["tn"], t["acc_overall"]),
    ]
    return pd.DataFrame(
        [(c, n, f"{k} ({_pct(a)})") for c, n, k, a in rows],
        columns=["Class", "Trees", "Classified correctly"],
    ).set_index("Class")

standalone_df(standalone)

[manual_forked] running 175 skeletons from /home/arun/PointClouds/SilvaFennica/code/test_data/ManuallyCorrectedStems/Forked_ALS_AdTree_Output
  [10/175]  last=ALS_gridID_10_7_plotID_na_treeID_317_skeleton.ply  status=ok
  [20/175]  last=ALS_gridID_12_17_plotID_na_treeID_33_skeleton.ply  status=ok
  [30/175]  last=ALS_gridID_14_19_plotID_na_treeID_96_skeleton.ply  status=ok


KeyboardInterrupt: 

Section 2 — segmentation quality and conditional detector accuracy (Tables 2 and 3):

In [ ]:
rows = []
for pipeline, stages in PIPELINES.items():
    inst_tab = stage_table(*stages["inst"])
    sem_tab  = stage_table(*stages["sem"])
    for cls, total in (("forked", config.TOTAL_FORKED),
                       ("unforked", config.TOTAL_UNFORKED)):
        r = funnel_row(cls, inst_tab, sem_tab, total)
        r["pipeline"] = pipeline
        rows.append(r)

def seg_survival_df(rows):
    data = [(r["pipeline"], CLASS_LABEL[r["class"]], r["total"],
             f'{r["instance_correct"]} ({_pct(r["instance_pct"])})',
             f'{r["stem_correct"]} ({_pct(r["stem_pct"])})')
            for r in rows]
    return pd.DataFrame(
        data, columns=["Pipeline", "Class", "Total",
                       "Instance correct", "Stem correct"]
    ).set_index(["Pipeline", "Class"])

def detector_conditional_df(rows):
    data = [(r["pipeline"], CLASS_LABEL[r["class"]],
             f'{r["det_given_inst"]} / {r["instance_correct"]} ({_pct(r["det_inst_pct"])})',
             f'{r["det_given_stem"]} / {r["stem_correct"]} ({_pct(r["det_stem_pct"])})')
            for r in rows]
    return pd.DataFrame(
        data, columns=["Pipeline", "Class",
                       "Given instance correct", "Given stem correct"]
    ).set_index(["Pipeline", "Class"])

display(seg_survival_df(rows))
display(detector_conditional_df(rows))

### 3. Base-trunk height vs MLS reference

For each setting, restrict to trees with an MLS reference, keep those detected
as forked, pair the predicted base-trunk height (`height_minmax`) against the
reference, and report RMSE, bias, MAE, R^2 and relative RMSE.

In [ ]:
lookup, mls_heights, _ = load_mls_reference()

SETTINGS = {
    "Manually corrected":                      "manual_forked",
    "ForestFormer3D (FOR-instanceV2)":         "p1_INST_forked",
    "ForestFormer3D (FGI-EMIT) + ResNet16FPN": "p2_INST_forked",
}

paired_by_setting = {}
for setting, ds in SETTINGS.items():
    recs = run_dataset(ds)
    recs = [r for r in recs if lookup(r["name"]) is not None]
    paired = pair_estimates_with_reference(
        recs, lookup, mls_heights, height_field="height_minmax")
    print_height_metrics(height_metrics(paired), title=setting)
    paired_by_setting[setting] = paired

Predicted against reference base-trunk height, one panel per setting. The
dashed line is 1:1.

In [ ]:
import matplotlib.pyplot as plt

settings = [s for s, p in paired_by_setting.items() if p]
fig, axes = plt.subplots(1, len(settings),
                         figsize=(4 * len(settings), 4),
                         sharex=True, sharey=True)
if len(settings) == 1:
    axes = [axes]

for ax, setting in zip(axes, settings):
    paired = paired_by_setting[setting]
    pred = [p for _, p, _ in paired]
    ref  = [r for _, _, r in paired]
    ax.scatter(ref, pred, s=18, alpha=0.7)
    lo = min(min(ref), min(pred))
    hi = max(max(ref), max(pred))
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_title(setting, fontsize=9)
    ax.set_xlabel("MLS reference (m)")

axes[0].set_ylabel("Predicted (m)")
fig.tight_layout()
plt.show()